# Cellular Senescence Transcriptome Analysis

Exploratory microarray analysis of Normal, Senescence, and Treatment conditions in human dermal fibroblasts. The private input workbook is not distributed.

## Objective
Identify genes with at least two-fold senescence-associated expression changes, evaluate movement toward the Normal expression state after Treatment, and examine GO Biological Process enrichment.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(ROOT))

from python.preprocessing import load_expression_data, ratio_log2_qc
from python.candidate_selection import select_candidates
from python.go_analysis import prepare_go_data, enrichment_by_direction, selective_recovery_test

DATA_PATH = ROOT / 'data' / 'expression_data.xlsx


## Data and QC
The workbook contains one sample per condition (n=1). Fold-change thresholds are therefore exploratory criteria, not gene-level significance tests.

In [ ]:
df = load_expression_data(DATA_PATH)
print(df.shape)
ratio_log2_qc(df)


## Candidate Selection
The workflow applies the original criteria: senescence-associated ≥2-fold change, movement closer to Normal after Treatment, return to 0.5–2.0-fold of Normal, and a stricter Treatment-response subgroup.

In [ ]:
selected = select_candidates(df)
summary = selected['summary']
summary

assert summary['senescence_changed'] == 718
assert summary['closer_to_normal'] == 330
assert summary['final_candidates'] == 181
assert summary['strong_candidates'] == 51


In [ ]:
final_candidates = selected['final_candidates']
strong_candidates = selected['strong_candidates']
final_candidates.head()


## GO Biological Process Analysis
GO enrichment uses the measured, BP-annotated genes as background, one-sided Fisher's exact tests, and Benjamini-Hochberg FDR correction.

In [ ]:
background_go, sen_go = prepare_go_data(df, selected['senescence_changed_probes'])
down_go = enrichment_by_direction(background_go, sen_go, 'Down')
up_go = enrichment_by_direction(background_go, sen_go, 'Up')

print('Down FDR < 0.05:', (down_go['FDR'] < 0.05).sum())
print('Up FDR < 0.05:', (up_go['FDR'] < 0.05).sum())


In [ ]:
down_recovery = selective_recovery_test(sen_go, down_go, final_candidates, 'Down')
up_recovery = selective_recovery_test(sen_go, up_go, final_candidates, 'Up')

print('Down selective recovery FDR < 0.05:', (down_recovery['FDR'] < 0.05).sum())
print('Up selective recovery FDR < 0.05:', (up_recovery['FDR'] < 0.05).sum() if not up_recovery.empty else 0)


## Results
The analysis identified 718 senescence-associated genes, 330 genes that moved closer to Normal after Treatment, 181 final recovery candidates, and a 51-gene strong Treatment-response subgroup. GO enrichment identified 60 FDR < 0.05 terms among Senescence-Down genes and 4 among Senescence-Up genes. No tested significant senescence GO term showed selective recovery at FDR < 0.05 after Treatment.

## Limitations
Each condition contains one sample (n=1), so gene-level statistical significance cannot be evaluated. Movement toward Normal is an expression-based criterion and does not establish functional restoration or causality.